# 🤖 WikipediaML - Google Colab Training

Bu notebook WikipediaML projesini Google Colab'da eğitmek için hazırlanmıştır.

## ⚠️ ÖNEMLİ NOTLAR:
- Her cell'i sırayla çalıştırın
- Eğitim 2-4 saat sürebilir
- Colab 12 saat sonra otomatik kapanır
- Model dosyalarını mutlaka indirin!

## 1️⃣ Proje Kurulumu

In [ ]:
from google.colab import files
import zipfile
import os

print("📁 WikipediaML.zip dosyasını seçin ve yükleyin...")
uploaded = files.upload()

print("\n📦 Dosyalar çıkartılıyor...")
!unzip -q WikipediaML.zip -d WikipediaML

# Klasöre geç
%cd WikipediaML

# Gerekli klasörleri oluştur
print("\n📁 Klasörler oluşturuluyor...")
!mkdir -p data cache

# Proje yapısını göster
print("\n✅ Proje yapısı:")
!ls -la

print("\n✅ src/ klasörü:")
!ls -la src/

print("\n🎉 Proje hazır!")

## 2️⃣ Dependencies Yükleme

In [ ]:
print("📦 Paketler yükleniyor...")

# Tüm gerekli paketler
!pip install -q xgboost scikit-learn sentence-transformers wikipedia-api beautifulsoup4 requests anthropic

print("\n✅ Tüm paketler yüklendi!")
print("\n📋 Yüklenen paketler:")
!pip list | grep -E 'xgboost|scikit-learn|sentence-transformers|wikipedia|anthropic'

## 3️⃣ Google Drive Bağlantısı (Opsiyonel - Önerilen)

**Neden Drive'a kaydetmeliyiz?**
- ✅ Colab kapansa bile veriler kaybolmaz
- ✅ Eğitimi durdursan bile devam edebilirsin
- ✅ Model dosyaları otomatik kaydedilir

**NOT:** İlk çalıştırmada izin isteyecek, "Allow" tıklayın.

In [ ]:
from google.colab import drive
import os

# Drive'ı bağla
print("📂 Google Drive bağlanıyor...")
drive.mount('/content/drive')

# WikipediaML klasörü oluştur
drive_path = '/content/drive/MyDrive/WikipediaML'
!mkdir -p {drive_path}/cache
!mkdir -p {drive_path}/data

# Symlink oluştur (cache/ klasörü Drive'a işaret etsin)
!rm -rf /content/WikipediaML/cache
!ln -s {drive_path}/cache /content/WikipediaML/cache

print(f"\n✅ Drive bağlandı: {drive_path}")
print("✅ cache/ klasörü Drive'a yönlendirildi")
print("\n💡 Artık tüm model dosyaları Drive'a kaydedilecek!")

# Kontrol et
!ls -la cache/

## 4️⃣ Mevcut Eğitimi Kontrol Et

**Eğer daha önce eğitim yaptıysan, devam edebilirsin!**

In [ ]:
import json
import os

# Mevcut eğitimi kontrol et
history_file = 'cache/training_history.json'

if os.path.exists(history_file):
    print("📊 Mevcut eğitim bulundu!\n")
    
    with open(history_file, 'r') as f:
        history = json.load(f)
    
    print(f"Toplam denemeler: {history['total_attempts']}")
    print(f"Başarılı: {history['successful_attempts']}")
    print(f"Başarı oranı: {history['success_rate']:.1f}%")
    print(f"Toplam süre: {history['total_training_time']:.1f}s ({history['total_training_time']/60:.1f} dakika)")
    
    if history['ml_model_trained']:
        print(f"\n✅ ML model eğitilmiş! ({history['ml_training_samples']} sample)")
        print("\n💡 Yeni eğitim yaparsanız, mevcut veriler üzerine eklenecek.")
    else:
        print(f"\n⚠️ ML model henüz eğitilmemiş (en az 10 başarılı path gerekli)")
    
    # Model dosyalarını kontrol et
    print("\n📁 Mevcut cache dosyaları:")
    !ls -lh cache/
else:
    print("📝 Yeni eğitim başlayacak (mevcut eğitim yok)")
    print("\n💡 İlk eğitim için 10-50 çift önerilir.")

## 5️⃣ Dataset Oluştur (Opsiyonel)

**Sadece yeni dataset oluşturmak istersen çalıştır.**

In [ ]:
# Büyük dataset oluştur (500 çift)
!python generate_large_dataset.py --count 500

# Dataset'i kontrol et
print("\n✅ Oluşturulan dataset:")
!ls -lh data/

## 6️⃣ Model Eğitimi

### ⚠️ ÖNEMLİ: Eğitim Süresi
- **10 çift:** 15-30 dakika (test için)
- **50 çift:** 1-2 saat (orta eğitim)
- **100 çift:** 2-4 saat (tam eğitim)

### 💡 İPUCU:
- İlk seferinde 10 çift ile test et
- Başarılı olursa 50-100 çift ile devam et
- Eğitim devam ederken Colab'ı kapatma!

In [ ]:
# Seçenek A: Küçük test (10 çift - 15-30 dakika) - ÖNERİLEN İLK ADIM
!python train_ml_model_curated.py --limit 10

# VEYA Seçenek B: Orta eğitim (50 çift - 1-2 saat)
# !python train_ml_model_curated.py --limit 50

# VEYA Seçenek C: Tam eğitim (100 çift - 2-4 saat)
# !python train_ml_model_curated.py --limit 100

## 7️⃣ Eğitim Durumunu İzle (Opsiyonel)

**Eğitim devam ederken başka bir cell'de çalıştırabilirsin.**

In [ ]:
import time
import json
from IPython.display import clear_output

print("📊 Eğitim izleniyor... (Durdurmak için Interrupt Runtime)\n")

while True:
    try:
        with open('cache/training_history.json', 'r') as f:
            history = json.load(f)
        
        clear_output(wait=True)
        
        print("📊 CANLI EĞİTİM İSTATİSTİKLERİ")
        print("="*50)
        print(f"Toplam denemeler: {history['total_attempts']}")
        print(f"Başarılı: {history['successful_attempts']}")
        print(f"Başarısız: {history['total_attempts'] - history['successful_attempts']}")
        print(f"Başarı oranı: {history['success_rate']:.1f}%")
        print(f"Geçen süre: {history['total_training_time']/60:.1f} dakika")
        print(f"ML model: {'✅ Eğitildi' if history['ml_model_trained'] else '⏳ Henüz değil'}")
        print("="*50)
        print(f"\n🔄 Son güncelleme: {time.strftime('%H:%M:%S')}")
        
        time.sleep(10)  # 10 saniyede bir güncelle
    except FileNotFoundError:
        print("⏳ Eğitim henüz başlamadı...")
        time.sleep(5)
    except KeyboardInterrupt:
        print("\n✅ İzleme durduruldu")
        break

## 8️⃣ Eğitim Sonuçlarını Kontrol Et

In [ ]:
import json

# Cache klasörünü kontrol et
print("📁 Cache dosyaları:")
!ls -lh cache/

# Training history'yi göster
print("\n📊 Training İstatistikleri:")
print("="*60)

with open('cache/training_history.json', 'r') as f:
    history = json.load(f)

print(f"Toplam denemeler: {history['total_attempts']}")
print(f"Başarılı: {history['successful_attempts']}")
print(f"Başarı oranı: {history['success_rate']:.1f}%")
print(f"Toplam süre: {history['total_training_time']:.1f}s ({history['total_training_time']/60:.1f} dakika)")
print(f"ML model eğitildi: {'✅ Evet' if history['ml_model_trained'] else '❌ Hayır'}")
print(f"Training samples: {history['ml_training_samples']}")

if history['ml_model_trained']:
    print("\n✅ Model başarıyla eğitildi!")
    print("📥 Şimdi model dosyalarını indirebilirsin (sonraki cell)")
else:
    print("\n⚠️ Model henüz eğitilmedi")
    print(f"   En az 10 başarılı path gerekli (şu an: {history['successful_attempts']})")
    print("   Daha fazla eğitim yapman gerekiyor.")

## 9️⃣ Model Dosyalarını İndir

**Eğitim tamamlandıktan sonra model dosyalarını indir.**

In [ ]:
from google.colab import files
import os

print("📥 Model dosyaları indiriliyor...\n")

# Ana model dosyaları
files_to_download = [
    'cache/ml_model.pkl',
    'cache/ml_scaler.pkl',
    'cache/training_history.json'
]

for file_path in files_to_download:
    if os.path.exists(file_path):
        files.download(file_path)
        print(f"✅ İndirildi: {file_path}")
    else:
        print(f"⚠️ Bulunamadı: {file_path}")

# Opsiyonel: Diğer cache dosyaları
print("\n📦 Opsiyonel cache dosyaları:")
optional_files = [
    'cache/embeddings_cache.pkl',
    'cache/wiki_graph.pkl',
    'cache/category_cache.pkl'
]

for file_path in optional_files:
    if os.path.exists(file_path):
        try:
            files.download(file_path)
            print(f"✅ İndirildi: {file_path}")
        except:
            print(f"⚠️ İndirilemedi: {file_path} (çok büyük olabilir)")
    else:
        print(f"⏭️ Atlandı: {file_path} (henüz oluşmadı)")

print("\n✅ İndirme tamamlandı!")
print("\n📝 Sonraki adımlar:")
print("1. İndirilen dosyaları lokal projenin cache/ klasörüne kopyala:")
print("   cp ~/Downloads/*.pkl cache/")
print("   cp ~/Downloads/*.json cache/")
print("\n2. Lokal'de test et:")
print("   python main.py --ml 'Albert_Einstein' 'Quantum_mechanics'")

## 🔟 Test Et (Opsiyonel - Colab'da)

**Model eğitildikten sonra Colab'da test edebilirsin.**

In [ ]:
# Colab'da test et
!python main.py --ml "Albert_Einstein" "Quantum_mechanics"

## 📊 Sistem Bilgileri

In [ ]:
# RAM kullanımı
print("💾 RAM Kullanımı:")
!free -h

# Disk kullanımı
print("\n💿 Disk Kullanımı:")
!df -h /content

# GPU durumu (eğer varsa)
print("\n🎮 GPU Durumu:")
!nvidia-smi 2>/dev/null || echo "GPU yok (CPU kullanılıyor)"

## ❓ Sık Sorulan Sorular

### 1. Eğitim ne kadar sürer?
- 10 çift: 15-30 dakika
- 50 çift: 1-2 saat
- 100 çift: 2-4 saat

### 2. Colab kapanırsa ne olur?
- Drive'a bağladıysan veriler kaybolmaz
- Tekrar başlatınca kaldığın yerden devam edersin

### 3. Tekrar eğitim yaparsam?
- Mevcut veriler üzerine eklenir
- Model daha da güçlenir
- Baştan başlamaz

### 4. Model dosyaları nerede?
- Drive'a bağladıysan: `MyDrive/WikipediaML/cache/`
- Bağlamadıysan: İndirmen gerekir

### 5. Hata alırsam?
- Runtime > Restart runtime
- Cell'leri tekrar çalıştır
- Veriler kaybolmaz (Drive'daysa)